**Navigation** : [carnet source Python](Constructeur-01-Lois-Comme-Taches-Python.ipynb) | [Index de la série](README.md)

# Théorie du constructeur — les lois comme tâches possibles ou impossibles (Rust)

**Série** : Physique / Constructeur · **Noyau** : Rust (evcxr), std seul

Jumeau Rust de [Constructeur-01 — les lois comme tâches possibles ou impossibles (Python)](Constructeur-01-Lois-Comme-Taches-Python.ipynb) : même fil pédagogique, mêmes entrées, mêmes verdicts — rejoués ici en **Rust**, bibliothèque standard seule, aucun crate externe.

La physique fondamentale s'énonce d'habitude en **dynamique** : des lois de mouvement qui
décrivent comment un état évolue, une fois les conditions initiales données. La
**théorie du constructeur** (Deutsch, 2013 ; Deutsch & Marletto, 2015) opère une inversion :
les lois fondamentales sont des énoncés sur **quelles transformations physiques sont
possibles, lesquelles sont impossibles, et pourquoi**.

> « Les lois sont sur lesquelles transformations physiques sont possibles et lesquelles
> sont impossibles, et pourquoi. » — constructortheory.org

Ce carnet distille ce changement de point de vue. Il est volontairement **conceptuel** :
les cellules de code illustrent les notions (tâche, constructeur, possible/impossible,
support d'information) sur des systèmes jouets, sans requérir de calcul lourd.

## Le vocabulaire : tâche, substrat, état, attribut, constructeur

Le site d'origine illustre avec un **dé** :

- **substrat** — l'objet qui est transformé (le dé) ;
- **état** — les six faces possibles du dé ;
- **attribut** — un ensemble d'états (« la face est 3 », « la face est paire ») ;
- **tâche** — la spécification d'une transformation du substrat, d'un attribut d'entrée
  vers un attribut de sortie (ici : 3 → 4) ;
- **constructeur** — « un objet qui peut accomplir la tâche **et qui reste le même**
  après l'avoir accomplie » : il conserve sa capacité à la produire, donc il la répète ;
- **possible / impossible** — une tâche est **impossible** si « aucun constructeur ne
  peut l'accomplir à une précision arbitraire » ; sinon elle est **possible**.

Subtilité importante (FAQ du site) : « les lois de la théorie du constructeur ne sont
**pas** des lois sur les constructeurs ». Le constructeur n'introduit que ce que signifie
« une tâche est possible ».

In [2]:
// Illustration 1 — une tâche sur un substrat (le dé)
// Le substrat a 6 états ; une tâche spécifie une transformation d'attribut.
use std::collections::BTreeMap;

fn py(b: bool) -> &'static str { if b { "True" } else { "False" } }

struct Tache { nom: String, entree: String, sortie: String }
impl Tache {
    fn new(nom: &str, entree: &str, sortie: &str) -> Self {
        Tache { nom: nom.to_string(), entree: entree.to_string(), sortie: sortie.to_string() }
    }
    fn accomplie_par(&self, constructeur: &Constructeur, etat: &str) -> bool {
        constructeur.transformer(etat) == self.sortie
    }
}

struct Constructeur { nom: String, regle: BTreeMap<String, String> }
impl Constructeur {
    fn new(nom: &str, regle: &[(&str, &str)]) -> Self {
        // regle : {entree: sortie}
        Constructeur { nom: nom.to_string(), regle: regle.iter().map(|(a, b)| (a.to_string(), b.to_string())).collect() }
    }
    fn transformer(&self, etat: &str) -> String {
        self.regle.get(etat).cloned().unwrap_or_else(|| etat.to_string())
    }
}

let _etats = ["1", "2", "3", "4", "5", "6"];
let tache = Tache::new("3 -> 4", "3", "4");
let c = Constructeur::new("c_3vers4", &[("3", "4")]);
println!("Tâche : {}", tache.nom);
println!("Accomplie par {} : {}", c.nom, py(tache.accomplie_par(&c, &tache.entree)));
println!("Un constructeur qui ne fait rien l'accomplit-il ? {}",
         py(tache.accomplie_par(&Constructeur::new("rien", &[]), &tache.entree)));

Tâche : 3 -> 4


Accomplie par c_3vers4 : True


Un constructeur qui ne fait rien l'accomplit-il ? False


## L'inversion : de la dynamique aux principes

La physique traditionnelle a pour objets « les trajectoires, les lois dynamiques et les
conditions initiales » : elle dit ce qui arrivera plus tard, sachant l'état actuel. La
théorie du constructeur renverse l'ordre — « les lois dynamiques et les trajectoires
arrivent bien plus tard » :

- les lois sont des **principes** : « quelles tâches sont possibles, lesquelles sont
  impossibles, et pourquoi » ;
- ils **sous-tendent et complètent** les théories fondamentales existantes, sans les
  remplacer ;
- la description traditionnelle (conditions initiales + lois de mouvement) **peut être
  retrouvée** dans de nombreux cas ;
- le gain : énoncer **exactement**, sans approximation, des domaines « considérés comme
  intrinsèquement approximatifs » — l'information, la vie, la probabilité, la
  thermodynamique.

Le motif : des entités comme l'information, la chaleur ou les propriétés du vivant
concernent « ce qui peut, ou non, être produit sur un système physique », pas ce qui
arrive sous des conditions initiales données.

## La conservation comme impossibilité

Dans ce langage, « la loi de conservation de l'énergie énonce que **certains
constructeurs sont impossibles** » — en particulier le perpétuum mobile de première
espèce : un constructeur qui rend du travail et revient à son état initial **sans**
apport d'énergie. La conséquence en découle « sans jamais référer à une machine à
mouvement perpétuel précise ».

In [3]:
// Illustration 2 — conservation de l'énergie = tâche impossible
// Un perpétuum mobile de 1re espèce est un constructeur qui rend du travail
// et revient à son état initial SANS apport d'énergie : impossible.

fn py(b: bool) -> &'static str { if b { "True" } else { "False" } }

fn est_perpetuum_mobile_1re_espece(apport_energie: i64, travail_rendu: i64) -> bool {
    apport_energie < travail_rendu
}

println!("Un moteur qui rend 10 J sans en recevoir est-il un perpétuum mobile ? {}", py(est_perpetuum_mobile_1re_espece(0, 10)));
println!("... et est-il donc possible ?  -> {}", py(!est_perpetuum_mobile_1re_espece(0, 10)));

Un moteur qui rend 10 J sans en recevoir est-il un perpétuum mobile ? True


... et est-il donc possible ?  -> False


## L'information comme support

L'information est définie **sans observateur**, par le **support d'information** :
« un système dont les états permettent **toutes les permutations** et aussi la **tâche
de copie** ». L'**interopérabilité** est le principe fort : deux systèmes qui sont chacun
un support d'information, composés, forment encore un support, de plus grande capacité.

C'est ce principe qui relie la théorie du constructeur au **no-cloning** : la copie d'un
état quantique inconnu est une tâche **impossible**, et le site la cite comme le
phénomène caractéristique que la théorie de l'information (Deutsch & Marletto, 2015)
ramène à une seule propriété constructeur.

In [4]:
// Illustration 3 — support d'information et no-cloning
// Support d'information : toutes les permutations possibles ET la copie possible.

fn py(b: bool) -> &'static str { if b { "True" } else { "False" } }

fn est_support_information(permutations_possibles: bool, copie_possible: bool) -> bool {
    permutations_possibles && copie_possible
}

println!("Bit classique (2 états, on peut tout permuter et copier) : {}", py(est_support_information(true, true)));
println!("Qubit inconnu (permutations ok, mais copie de l'inconnu IMPOSSIBLE) : {}", py(est_support_information(true, false)));

Bit classique (2 états, on peut tout permuter et copier) : True


Qubit inconnu (permutations ok, mais copie de l'inconnu IMPOSSIBLE) : False


## Le pont vers la série et au-delà

- **→ `Jauge/`** : le théorème de Noether associe à chaque symétrie continue une quantité
  conservée. Lue en langage constructeur, une conservation est un énoncé de **tâche
  impossible** (certains constructeurs sont exclus) — le même type de principe que la
  conservation de l'énergie ci-dessus.
- **→ `IIT/ICT-Series/` (causal bridges)** : le principe de **localité** du site — « une
  transformation qui agit sur A *seul* ne peut changer B » — est le contrepoint
  constructeur de la structure causale ; le site insiste sur la localité (pas
  « causalité ») et rejette le vocabulaire de « non-localité » de type Bell.
- **→ hybridation quantique** : le cas d'école du site est la **gravité** : « si une
  masse est quantique, elle peut être en deux lieux à la fois, et cela brise la
  structure de la relativité générale ». Un médiateur classique ne peut pas créer
  d'intrication — un test positif invaliderait l'interopérabilité **ou** la localité.

## Le test de la gravité en simulation : le témoin de non-classicité

Le cas d'école du pont ci-dessus — la **gravité** comme médiateur — est aussi le
test le plus abouti de la théorie du constructeur. Le **théorème général des
témoins** (General Witness Theorem) s'énonce ainsi : si un **médiateur** M (par
exemple la gravité) parvient à **intriquer** deux systèmes quantiques Q_A et Q_B
par des **interactions locales** (Q_A ne parle qu'à M, M à Q_B, jamais Q_A à Q_B
directement), alors M **ne peut pas être classique**. Le **test BMV**
(Bose-Marletto-Vedral) en est l'application : deux masses, chacune dans une
superposition de positions, reliées seulement par leur champ gravitationnel ; une
intrication observée entre elles rendrait la gravité non-classique.

Ces tests sont recensés et discutés par Marletto, Deutsch & Vedral dans *Tests of
constructor theory* (2606.07352), qui y passe aussi en revue une **simulation**
du modèle BMV (qubits NMR, Jones et al.) suivant la transition d'un comportement
entièrement quantique (qubit) à un comportement entièrement classique (bit). Nous
reproduisons ici cette simulation, en Rust, std seul :

In [5]:
// Illustration 4 — le test BMV de la gravité, en simulation (témoin de non-classicité)
// Deux sondes quantiques (qubits), chacune dans une superposition de deux positions.
// Un mediateur M les relie ; chacune n'interagit qu'avec M (jamais entre elles).
// Theoreme general (2606.07352) : si M intrique les deux sondes par de telles
// interactions locales, M n'est PAS classique.

// Valeurs propres d'une matrice symetrique reelle 4x4 par la methode de Jacobi cyclique, std seul :
// une rotation par paire (p, q), balayages repetes tant que la somme des carres hors diagonale
// reste au-dessus de 1e-60 (100 balayages au plus) ; valeurs rendues par ordre croissant, comme
// `numpy.linalg.eigvalsh`. A recopier tel quel : la sonde le compare, bit pour bit, a son miroir Python.
fn eigvalsh_jacobi(m: [[f64; 4]; 4]) -> [f64; 4] {
    let mut a = m;
    for _ in 0..100 {
        let mut off = 0.0;
        for p in 0..4 {
            for q in (p + 1)..4 {
                off += a[p][q] * a[p][q];
            }
        }
        if off < 1e-60 {
            break;
        }
        for p in 0..4 {
            for q in (p + 1)..4 {
                let apq = a[p][q];
                if apq == 0.0 {
                    continue;
                }
                let theta = (a[q][q] - a[p][p]) / (2.0 * apq);
                let mut t = if theta.abs() > 1e150 {
                    0.5 / theta.abs()
                } else {
                    1.0 / (theta.abs() + (theta * theta + 1.0).sqrt())
                };
                if theta < 0.0 {
                    t = -t;
                }
                let c = 1.0 / (t * t + 1.0).sqrt();
                let s = t * c;
                let (app, aqq) = (a[p][p], a[q][q]);
                a[p][p] = app - t * apq;
                a[q][q] = aqq + t * apq;
                a[p][q] = 0.0;
                a[q][p] = 0.0;
                for k in 0..4 {
                    if k != p && k != q {
                        let (akp, akq) = (a[k][p], a[k][q]);
                        a[k][p] = c * akp - s * akq;
                        a[p][k] = a[k][p];
                        a[k][q] = s * akp + c * akq;
                        a[q][k] = a[k][q];
                    }
                }
            }
        }
    }
    let mut v = [a[0][0], a[1][1], a[2][2], a[3][3]];
    v.sort_by(|x, y| x.total_cmp(y));
    v
}

type M4 = [[f64; 4]; 4];

fn etat_x() -> [f64; 2] {
    let r = 2.0f64.sqrt();
    [1.0 / r, 1.0 / r] // superposition des 2 positions
}

fn rho_probes() -> M4 {
    let x = etat_x();
    let mut v = [0.0; 4];
    for i in 0..2 { for j in 0..2 { v[2 * i + j] = x[i] * x[j]; } } // |x+>|x+>  (produit, matrice 4x4)
    let mut m = [[0.0; 4]; 4];
    for i in 0..4 { for j in 0..4 { m[i][j] = v[i] * v[j]; } }
    m
}

fn matmul(a: &M4, b: &M4) -> M4 {
    let mut m = [[0.0; 4]; 4];
    for i in 0..4 { for j in 0..4 { let mut acc = 0.0; for k in 0..4 { acc += a[i][k] * b[k][j]; } m[i][j] = acc; } }
    m
}

fn combine(ka: f64, a: &M4, kb: f64, b: &M4) -> M4 {
    let mut m = [[0.0; 4]; 4];
    for i in 0..4 { for j in 0..4 { m[i][j] = ka * a[i][j] + kb * b[i][j]; } }
    m
}

fn diag(d: [f64; 4]) -> M4 {
    let mut m = [[0.0; 4]; 4];
    for i in 0..4 { m[i][i] = d[i]; }
    m
}

fn negativite(rho: &M4) -> f64 {
    // T partiel sur B : rt[2a+x][2c+y] = rho[2a+y][2c+x]
    let mut rt = [[0.0; 4]; 4];
    for a in 0..2 { for x in 0..2 { for c in 0..2 { for y in 0..2 { rt[2 * a + x][2 * c + y] = rho[2 * a + y][2 * c + x]; } } } }
    let mut somme = 0.0;
    for v in eigvalsh_jacobi(rt) { somme += v.abs(); }
    let n = 0.5 * (somme - 1.0);
    if n > 0.0 { n } else { 0.0 } // max(0.0, n) de Python
}

const SEUIL_INTRICATION: f64 = 0.0; // seuil : une negativite strictement positive => intrication

fn verdict_non_classicite(n: f64) -> &'static str {
    if n > SEUIL_INTRICATION { "NON-CLASSIQUE" } else { "classique" }
}

let cz = diag([1.0, 1.0, 1.0, -1.0]); // mediateur QUANTIQUE : interaction de type Ising
let zz = diag([1.0, -1.0, -1.0, 1.0]); // phase locale commune (classique)

// 1) Mediateur quantique : l'interaction intrique les deux sondes.
let rho_q = matmul(&matmul(&cz, &rho_probes()), &cz);
let n_quantique = negativite(&rho_q);

// 2) Mediateur classique (bit) : transmet une phase locale commune, moyennee sur
// l'inconnue classique (50/50) -> correlation classique, PAS d'intrication.
let rho_classique = combine(0.5, &rho_probes(), 0.5, &matmul(&matmul(&zz, &rho_probes()), &zz));
let n_classique = negativite(&rho_classique);

println!("Temoin de non-classicite (2606.07352) : le mediateur intrique-t-il les sondes ?");
println!("  mediateur quantique : negativite = {:.3}  -> {}", n_quantique, verdict_non_classicite(n_quantique));
println!("  mediateur classique : negativite = {:.3}  -> {}", n_classique, verdict_non_classicite(n_classique));
println!();
println!("Transition qubit -> bit (le mediateur perd sa coherence quantique) :");
for p in [0.0, 0.25, 0.5, 0.75, 1.0] {
    let rho = combine(1.0 - p, &rho_q, p, &rho_classique);
    println!("  part classique p={:.2} : negativite = {:.3}", p, negativite(&rho));
};

Temoin de non-classicite (2606.07352) : le mediateur intrique-t-il les sondes ?


  mediateur quantique : negativite = 0.500  -> NON-CLASSIQUE


  mediateur classique : negativite = 0.000  -> classique


Transition qubit -> bit (le mediateur perd sa coherence quantique) :


  part classique p=0.00 : negativite = 0.500


  part classique p=0.25 : negativite = 0.318


  part classique p=0.50 : negativite = 0.155


  part classique p=0.75 : negativite = 0.038


  part classique p=1.00 : negativite = 0.000


## Exercices

In [6]:
// Exercice 1 — accomplir une tâche
// Retourne l'état de sortie si le constructeur amène `tache.entree` vers
// `tache.sortie`, sinon None (tâche non accomplie par ce constructeur).
use std::collections::BTreeMap;

struct Tache { nom: String, entree: String, sortie: String }
impl Tache {
    fn new(nom: &str, entree: &str, sortie: &str) -> Self {
        Tache { nom: nom.to_string(), entree: entree.to_string(), sortie: sortie.to_string() }
    }
    fn accomplie_par(&self, constructeur: &Constructeur, etat: &str) -> bool {
        constructeur.transformer(etat) == self.sortie
    }
}

struct Constructeur { nom: String, regle: BTreeMap<String, String> }
impl Constructeur {
    fn new(nom: &str, regle: &[(&str, &str)]) -> Self {
        Constructeur { nom: nom.to_string(), regle: regle.iter().map(|(a, b)| (a.to_string(), b.to_string())).collect() }
    }
    fn transformer(&self, etat: &str) -> String {
        self.regle.get(etat).cloned().unwrap_or_else(|| etat.to_string())
    }
}

fn accomplir(_tache: &Tache, _constructeur: &Constructeur) -> Option<String> {
    // TODO etudiant
    None
}

println!("Exercice a completer");

Exercice a completer


In [7]:
// Exercice 2 — est-ce un constructeur ?
// Un constructeur « reste le même » après la tâche : sa capacité à transformer est
// conservée. Retourne True si l'objet est inchangé, False sinon, None si non défini.

fn est_constructeur(_etat_avant: &str, _etat_apres: &str) -> Option<bool> {
    // TODO etudiant
    None
}

println!("Exercice a completer");

Exercice a completer


In [8]:
// Exercice 3 — l'interopérabilité
// Deux supports d'information composés forment-ils encore un support ?
// (indice : c'est le principe d'interopérabilité de l'Illustration 3)

fn interoperable(_support_a: bool, _support_b: bool) -> Option<bool> {
    // TODO etudiant
    None
}

println!("Exercice a completer");

Exercice a completer


## Ce que ce carnet ne couvre pas

Restitué fidèlement au site : la **mécanique** du no-cloning (son énoncé et sa preuve
requièrent la théorie quantique — cf. le papier de l'information), la **deuxième loi**
de la thermodynamique (la « théorie du constructeur de la thermodynamique » est annoncée
mais non développée sur le site), et la **causalité** en tant que telle (le site ne
traite que la localité).

## À retenir

| Notion | Ce qu'elle dit | Où la voir |
|---|---|---|
| Tâche et constructeur | une tâche est une transformation d'attribut (3 → 4 sur le dé) ; un constructeur l'accomplit **et reste le même**, donc il peut la répéter : `c_3vers4` l'accomplit (`True`), le constructeur « rien » non (`False`) | Illustration 1 |
| Possible / impossible | une tâche est impossible si **aucun** constructeur ne peut l'accomplir à une précision arbitraire ; les lois de la théorie portent sur les tâches, pas sur les constructeurs | Le vocabulaire |
| Inversion | les lois deviennent des principes sur ce qui peut, ou non, être produit ; la description dynamique (conditions initiales et lois de mouvement) se retrouve dans de nombreux cas, elle n'est pas remplacée | L'inversion |
| Conservation | la conservation de l'énergie exclut un constructeur : le moteur qui rend 10 J sans en recevoir est un perpétuum mobile de première espèce (`True`), donc impossible (`False`) | Illustration 2 |
| Support d'information | permutations **et** copie possibles : le bit classique en est un (`True`), le qubit inconnu non (`False`), parce que copier un état quantique inconnu est une tâche impossible (no-cloning) | Illustration 3 |
| Témoin de non-classicité | un médiateur qui intrique deux sondes par des interactions seulement locales n'est pas classique : négativité 0,500 avec le médiateur quantique, 0,000 avec le médiateur classique, et 0,318 · 0,155 · 0,038 quand la part classique $p$ passe de 0,25 à 0,75 | Illustration 4 |

Les quatre illustrations sont des **systèmes jouets** : elles mettent le vocabulaire en action, elles ne démontrent pas les principes. Les booléens des illustrations 1 à 3 suivent directement des définitions codées ; seule l'illustration 4 calcule une quantité (la négativité, par transposée partielle) dont la valeur n'est pas écrite d'avance dans le code — et l'étiquette de non-classicité qui l'accompagne en est **déduite du seuil** (négativité strictement positive), non écrite d'avance non plus.

## Vérifiez votre compréhension

1. Le constructeur « rien » (règle vide) n'accomplit pas la tâche 3 → 4. Pourquoi cela ne suffit-il **pas** à dire que la tâche est impossible ?

<details><summary>Réponse</summary>

Parce qu'« impossible » signifie qu'**aucun** constructeur ne peut accomplir la tâche à une précision arbitraire. Un échec particulier ne dit rien de tous les constructeurs : `c_3vers4` l'accomplit (`True`), la tâche est donc possible. L'impossibilité est un énoncé sur la classe entière des constructeurs : c'est pour cela que la théorie l'énonce comme une loi (conservation, no-cloning), et non comme le résultat d'un essai.

</details>

2. En quoi la conservation de l'énergie, dans ce langage, ressemble-t-elle au no-cloning ?

<details><summary>Réponse</summary>

Les deux sont des énoncés de **tâche impossible**. L'une exclut le constructeur qui rend du travail sans apport d'énergie (illustration 2 : possible → `False`) ; l'autre exclut la copie d'un état quantique inconnu (illustration 3 : le qubit inconnu n'est pas un support d'information → `False`). Aucun des deux ne décrit une trajectoire : tous deux disent ce qui ne peut pas être produit, quelle que soit la machine envisagée. C'est aussi la lecture que le pont vers `Jauge/` propose pour les conservations de Noether.

</details>

3. Dans la simulation BMV, que montre la décroissance de la négativité de 0,500 à 0,000 quand la part classique $p$ passe de 0 à 1, et que ne montre-t-elle **pas** ?

<details><summary>Réponse</summary>

Elle montre que l'intrication des sondes s'efface à mesure que le médiateur perd sa cohérence quantique : un médiateur entièrement classique ($p = 1$) n'en crée aucune, comme l'exige le théorème général des témoins. La transition est progressive (0,318 · 0,155 · 0,038) : tant que la négativité reste strictement positive, les sondes sont intriquées, et le théorème interdit qu'un médiateur entièrement classique l'ait produit. Ce qu'elle ne montre pas : que la gravité est quantique. Le médiateur est ici un modèle choisi (une interaction de type Ising, `CZ`), pas la gravité ; seule une expérience BMV réelle pourrait trancher.

</details>

## Pour aller plus loin

- **Sous-série `Jauge/`** : [Jauge-01 — Théorème de Noether](../Jauge/Jauge-01-Theoreme-Noether-Python.ipynb) — la conservation y est **mesurée** (courant de Noether, dérive numérique, contre-témoin anisotrope) ; la relire ensuite en langage constructeur, comme un énoncé de tâche impossible.
- **Localité et structure causale** : [IIT / ICT-Series](../../IIT/ICT-Series/README.md) — les trajectoires de structures causales, contrepoint du principe de localité du site (« une transformation qui agit sur A *seul* ne peut changer B »).
- **Jumeau Python** : [Constructeur-01 — les lois comme tâches (Python)](Constructeur-01-Lois-Comme-Taches-Python.ipynb) — même plan, mêmes valeurs ; les deux carnets doivent rendre la même négativité.
- **Plan de la sous-série** : [README Constructeur](README.md).
- **Fil de la série** : [#23161 — EPIC Série Physique numérique](https://github.com/jsboige/CoursIA/issues/23161).

## Références

- D. Deutsch, *The Philosophy of Constructor Theory*, **Synthese** 190(18), 2013.
  [doi:10.1007/s11229-013-0279-z](https://doi.org/10.1007/s11229-013-0279-z) ·
  [arXiv:1210.7439](https://arxiv.org/abs/1210.7439)
- D. Deutsch, C. Marletto, *The Constructor Theory of Information*,
  **Proc. R. Soc. A** 471(2174), 2015.
  [doi:10.1098/rspa.2014.0540](https://doi.org/10.1098/rspa.2014.0540) ·
  [arXiv:1405.5563](https://arxiv.org/abs/1405.5563)
- C. Marletto, D. Deutsch, V. Vedral, *Tests of constructor theory*, 2026.
  [arXiv:2606.07352](https://arxiv.org/abs/2606.07352)
- Site du projet : <https://www.constructortheory.org/> (pages *What is*, *FAQ*,
  *Applications* — lues et ancrées pour ce carnet).

> Carnet issu de l'EPIC #26534.